## (1) Load model

In [1]:
from model import Mamba, ModelArgs
from transformers import AutoTokenizer

# One of:
#     'state-spaces/mamba-2.8b-slimpj'
#     'state-spaces/mamba-2.8b'
#     'state-spaces/mamba-1.4b'
#     'state-spaces/mamba-790m'
#     'state-spaces/mamba-370m'
#     'state-spaces/mamba-130m'
pretrained_model_name = 'state-spaces/mamba-370m'

model = Mamba.from_pretrained(pretrained_model_name)
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neox-20b')

c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## (2) Generate Text

In [2]:
import torch
import torch.nn.functional as F


def generate(model,
             tokenizer,
             prompt: str,
             n_tokens_to_gen: int = 50,
             sample: bool = True,
             top_k: int = 40):
    model.eval()
    
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids
    
    for token_n in range(n_tokens_to_gen):
        with torch.no_grad():
            indices_to_input = input_ids
            next_token_logits = model(indices_to_input)[:, -1]
        
        probs = F.softmax(next_token_logits, dim=-1)
        (batch, vocab_size) = probs.shape
        
        if top_k is not None:
            (values, indices) = torch.topk(probs, k=top_k)
            probs[probs < values[:, -1, None]] = 0
            probs = probs / probs.sum(axis=1, keepdims=True)
        
        if sample:
            next_indices = torch.multinomial(probs, num_samples=1)
        else:
            next_indices = torch.argmax(probs, dim=-1)[:, None]
        
        input_ids = torch.cat([input_ids, next_indices], dim=1)

    output_completions = [tokenizer.decode(output.tolist()) for output in input_ids][0]
    
    return output_completions

In [3]:
print(generate(model, tokenizer, 'Mamba is the'))

Mamba is the biggest challenge, I like to think we have the opportunity to get out to a good level this year."

There is a lot to work on, with a high number of performances by the new guys to be played in the team's set-


In [4]:
print(generate(model, tokenizer, 'John: Hi!\nSally:'))

John: Hi!
Sally: It's been forever since I did a video.
John: I missed you from the past two weeks.
Sally: I had to come back in the same time slot I was
going to for my show at L'Affaire.


In [5]:
print(generate(model, tokenizer, 'The meaning of life is '))

The meaning of life is （e.g. the meaning of life is） 《東》 is a Japanese poetry written in the Meiji period (1868–1926) and composed by Shinsai Nakajima and written by Chikuzon


In [6]:
print(generate(model, tokenizer, 'def reverse_string('))

def reverse_string(self, prefix):
        r = ''
        for r in self.to_string_list():
            if r[0] not in prefix and r[1] not in prefix:
                break
            l = r.
